# 03 — Prepare CASIA-B Splits

Creates ST, MT, LT train/test/gallery/probe CSVs.

In [1]:
from pathlib import Path
import os, random, time, gc, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

EXP_DIR = Path('/media/wadud/DriveUbuntu/GaitRecognition 2.0')
POSE_TAG = None  # None = auto-detect folder under data/poses with most .npz files
POSES_DIR = EXP_DIR / 'data' / 'poses'
SPLIT_DIR = EXP_DIR / 'data' / 'splits'
REPORT_DIR = EXP_DIR / 'data' / 'reports'
RESULT_DIR = EXP_DIR / 'results'
CHECKPOINT_DIR = EXP_DIR / 'checkpoints'
LOG_DIR = EXP_DIR / 'logs'
for d in [SPLIT_DIR, REPORT_DIR, RESULT_DIR, CHECKPOINT_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def detect_pose_root(poses_dir=POSES_DIR, pose_tag=POSE_TAG):
    if pose_tag is not None:
        root = poses_dir / pose_tag
        if not root.exists():
            raise FileNotFoundError(root)
        return root
    candidates = [p for p in poses_dir.iterdir() if p.is_dir()] if poses_dir.exists() else []
    if not candidates:
        raise FileNotFoundError(f'No pose folders under {poses_dir}')
    counts = sorted([(len(list(p.rglob('*.npz'))), p) for p in candidates], reverse=True, key=lambda x: x[0])
    return counts[0][1]

POSE_ROOT = detect_pose_root()
print('EXP_DIR  :', EXP_DIR)
print('POSE_ROOT:', POSE_ROOT)

EXP_DIR  : /media/wadud/DriveUbuntu/GaitRecognition 2.0
POSE_ROOT: /media/wadud/DriveUbuntu/GaitRecognition 2.0/data/poses/yolo26l_pose


In [2]:
def rec(p):
    rel=p.relative_to(POSE_ROOT); subject=rel.parts[0]; condition,seq=rel.parts[1].split('-'); view=p.stem
    return {'pose_path':str(p),'subject':subject,'condition':condition,'seq':seq,'view':view}
df_all=pd.DataFrame([rec(p) for p in sorted(POSE_ROOT.rglob('*.npz'))])
df_all.to_csv(SPLIT_DIR/'all_pose_files.csv',index=False)
print('All pose files:',len(df_all)); display(df_all.head()); display(df_all.groupby(['condition','seq']).size().reset_index(name='count'))

All pose files: 13640


,pose_path,subject,condition,seq,view
0,/media/wadud/DriveUbuntu/GaitRecognition 2.0/d...,001,bg,01,000
1,/media/wadud/DriveUbuntu/GaitRecognition 2.0/d...,001,bg,01,018
2,/media/wadud/DriveUbuntu/GaitRecognition 2.0/d...,001,bg,01,036
3,/media/wadud/DriveUbuntu/GaitRecognition 2.0/d...,001,bg,01,054
4,/media/wadud/DriveUbuntu/GaitRecognition 2.0/d...,001,bg,01,072


,condition,seq,count
0,bg,01,1364
1,bg,02,1364
2,cl,01,1364
3,cl,02,1364
4,nm,01,1364
5,nm,02,1364
6,nm,03,1364
7,nm,04,1364
8,nm,05,1364
9,nm,06,1364


In [3]:
SPLITS={
 'ST':([f'{i:03d}' for i in range(1,25)],[f'{i:03d}' for i in range(25,125)]),
 'MT':([f'{i:03d}' for i in range(1,63)],[f'{i:03d}' for i in range(63,125)]),
 'LT':([f'{i:03d}' for i in range(1,75)],[f'{i:03d}' for i in range(75,125)]),
}
for name,(train_subs,test_subs) in SPLITS.items():
    (SPLIT_DIR/f'train_{name}_subjects.txt').write_text('\n'.join(train_subs)+'\n')
    (SPLIT_DIR/f'test_{name}_subjects.txt').write_text('\n'.join(test_subs)+'\n')
    train=df_all[df_all.subject.isin(train_subs)].copy(); test=df_all[df_all.subject.isin(test_subs)].copy()
    gallery=test[(test.condition=='nm') & (test.seq.isin(['01','02','03','04']))].copy()
    p_nm=test[(test.condition=='nm') & (test.seq.isin(['05','06']))].copy()
    p_bg=test[(test.condition=='bg') & (test.seq.isin(['01','02']))].copy()
    p_cl=test[(test.condition=='cl') & (test.seq.isin(['01','02']))].copy()
    train.to_csv(SPLIT_DIR/f'train_{name}.csv',index=False); test.to_csv(SPLIT_DIR/f'test_{name}.csv',index=False)
    gallery.to_csv(SPLIT_DIR/f'gallery_{name}.csv',index=False); p_nm.to_csv(SPLIT_DIR/f'probe_{name}_nm.csv',index=False)
    p_bg.to_csv(SPLIT_DIR/f'probe_{name}_bg.csv',index=False); p_cl.to_csv(SPLIT_DIR/f'probe_{name}_cl.csv',index=False)
    overlap=set(train.subject.unique()) & set(test.subject.unique())
    print('\n',name,'overlap:',len(overlap),'train:',len(train),'gallery:',len(gallery),'probe_nm:',len(p_nm),'probe_bg:',len(p_bg),'probe_cl:',len(p_cl))
print('Saved splits to:',SPLIT_DIR)


 ST overlap: 0 train: 2640 gallery: 4400 probe_nm: 2200 probe_bg: 2200 probe_cl: 2200

 MT overlap: 0 train: 6820 gallery: 2728 probe_nm: 1364 probe_bg: 1364 probe_cl: 1364

 LT overlap: 0 train: 8140 gallery: 2200 probe_nm: 1100 probe_bg: 1100 probe_cl: 1100
Saved splits to: /media/wadud/DriveUbuntu/GaitRecognition 2.0/data/splits
